In [6]:
# Imports

import sys
import os

import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

# Suppress error messages
stderr = sys.stderr
sys.stderr = open(os.devnull, 'w')

from transformers import (
    AutoTokenizer, 
    AutoModel,
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
)
# Restore error messages
sys.stderr = stderr

from su_utils import deserialize_tuple, make_distribution_target, eval_distribution, compute_distribution_metrics

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

#Kaggle doesn't allow importing this right now
#from dist_model import BertForDistributionPrediction, DistributionDataset, DistributionDataCollator

print('Imports ok.')

Imports ok.


In [7]:
## Debug/Check output dir
#!ls /kaggle/
#!head -n30 /kaggle/usr/lib/su_utils/su_utils.py

In [8]:
#dist_model.py 
# Workaround since Kaggle is acting weird. This is meant as a separate script
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from scipy.spatial.distance import cosine, jensenshannon

from transformers import (
    AutoTokenizer, 
    AutoModel,
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
)

print('Imports ok.')
from datasets import Dataset as HFDataset

#===============================================#
# BERT with Softmax for Distribution Prediction #
#===============================================#
class BertForDistributionPrediction(nn.Module):
    """
    BERT model with softmax output for predicting probability distributions.
    """
    def __init__(self, model_path, num_labels, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_path, local_files_only=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.num_labels = num_labels
        
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use [CLS] token representation
        pooled = outputs.last_hidden_state[:, 0, :]  # (batch, hidden_size)
        pooled = self.dropout(pooled)
        
        logits = self.classifier(pooled)  # (batch, num_labels)
        
        # Apply softmax to get probability distribution
        probs = torch.softmax(logits, dim=-1)  # (batch, num_labels)
        
        loss = None
        if labels is not None:
            # KL Divergence loss: KL(target || predicted)
            # Add small epsilon to avoid log(0)
            loss = nn.functional.kl_div(
                torch.log(probs + 1e-10),
                labels,
                reduction='batchmean'
            )
        
        return {"loss": loss, "logits": probs}

#=====================================#
# CUSTOM DATASET CLSS & DATA COLLATOR #
#=====================================#

class DistributionDataset(torch.utils.data.Dataset):
    """PyTorch dataset for distribution prediction."""
    def __init__(self, texts, distributions, tokenizer, max_length=512):
        self.texts = texts
        self.distributions = distributions
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors=None
        )
        
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "labels": self.distributions[idx].astype(np.float32)
        }


class DistributionDataCollator:
    """CCollator that handles distribution labels."""
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    
    def __call__(self, features):
        # Separate labels from input features
        labels = [f["labels"] for f in features]
        
        # Prepare inputs for tokenizer padding
        input_features = [{
            "input_ids": f["input_ids"],
            "attention_mask": f["attention_mask"]
        } for f in features]
        
        # Pad inputs
        batch = self.tokenizer.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )
        
        # Add labels
        batch["labels"] = torch.tensor(np.array(labels), dtype=torch.float32)
        
        return batch

Imports ok.


In [9]:
#Load and prepare data
data_dir = Path("/kaggle/input/prepreprocessed-for-bert")  # Adjust as needed
model_dir = Path("/kaggle/input/kbbert-swe/kbbert_swe")    # Adjust as needed

train_ml_export = pd.read_csv(data_dir / "train_ml_export.csv")
val_ml_export   = pd.read_csv(data_dir / "val_ml_export.csv")
label_list      = joblib.load(data_dir / "uo_label_list.joblib")
num_labels      = len(label_list)

print(f"Train: {len(train_ml_export)}, Val: {len(val_ml_export)}")
print(f"Labels ({num_labels}): {label_list}")

# Deserialize the tuple columns
train_ml_export["labels_uo"]  = train_ml_export["labels_uo"].apply(deserialize_tuple)
train_ml_export["labels_pct"] = train_ml_export["labels_pct"].apply(deserialize_tuple)
val_ml_export["labels_uo"]    = val_ml_export["labels_uo"].apply(deserialize_tuple)
val_ml_export["labels_pct"]   = val_ml_export["labels_pct"].apply(deserialize_tuple)



Train: 7865, Val: 1905
Labels (10): [2434, 2436, 2438, 2439, 2441, 2442, 2444, 2445, 2447, 2451]


In [10]:
# Build distributional target labels

# Build index mapping
uo_to_idx = {uo: i for i, uo in enumerate(label_list)}

# Create distribution targets
Y_train_dist = np.vstack([
    make_distribution_target(row, uo_to_idx, num_labels)
    for _, row in train_ml_export.iterrows()
])

Y_val_dist = np.vstack([
    make_distribution_target(row, uo_to_idx, num_labels)
    for _, row in val_ml_export.iterrows()
])

print(f"\nDistribution targets shape: train={Y_train_dist.shape}, val={Y_val_dist.shape}")
print(f"Sample train distribution (first 3):\n{Y_train_dist[:3]}")
print(f"Sum check (should be 1.0): {Y_train_dist[:3].sum(axis=1)}")


Distribution targets shape: train=(7865, 10), val=(1905, 10)
Sample train distribution (first 3):
[[0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]]
Sum check (should be 1.0): [1. 1. 1.]


In [11]:
#Prepare Training

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)

# Prepare texts
train_texts = train_ml_export["text"].astype(str).tolist()
val_texts   = val_ml_export["text"].astype(str).tolist()

# Create datasets
train_dataset = DistributionDataset(train_texts, Y_train_dist, tokenizer)
val_dataset   = DistributionDataset(val_texts, Y_val_dist, tokenizer)

# Initialize model
model = BertForDistributionPrediction(model_dir, num_labels)
print(f"Model initialized with {num_labels} output labels")

# Data collator
data_collator = DistributionDataCollator(tokenizer)

# Training arguments
training_args = TrainingArguments(
    output_dir="./kbbert_distribution",
    report_to="none",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="mae_pct",
    greater_is_better=False,  # Lower MAE is better
    save_total_limit=2,
    fp16=True,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_distribution_metrics,
)

print("Trainer initialized. Ready to train.")


Model initialized with 10 output labels
Trainer initialized. Ready to train.


In [12]:
# Perform training
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

trainer.train()

# Final evaluation
final_metrics = trainer.evaluate()
print("\n" + "="*60)
print("FINAL EVALUATION METRICS")
print("="*60)
for k, v in final_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")


STARTING TRAINING


Epoch,Training Loss,Validation Loss,Mae Pct,Top1 Accuracy,Mean Cosine Sim,Mean Js Divergence
1,0.186400,0.221847,1.892336,0.914961,0.942684,0.115545
2,0.121400,0.206116,1.588647,0.921260,0.949788,0.090396
3,0.035000,0.215835,1.436711,0.923885,0.951079,0.079450



FINAL EVALUATION METRICS
  eval_loss: 0.2158
  eval_mae_pct: 1.4367
  eval_top1_accuracy: 0.9239
  eval_mean_cosine_sim: 0.9511
  eval_mean_js_divergence: 0.0794
  eval_runtime: 35.3052
  eval_samples_per_second: 53.9580
  eval_steps_per_second: 1.6990
  epoch: 3.0000


In [13]:
# =============================================================================
# GET PREDICTIONS FROM TRAINED MODEL
# =============================================================================
print("\n" + "="*60)
print("GENERATING PREDICTIONS")
print("="*60)

# Get predictions from the model
pred_output = trainer.predict(val_dataset)
pred_dist = pred_output.predictions  # Already softmax (sum to 1)

# Scale to percentages for interpretability
pred_pct = pred_dist * 100

print(f"Predictions shape: {pred_pct.shape}")
print(f"Sample prediction (sums to ~100): {pred_pct[0].sum():.2f}")



GENERATING PREDICTIONS
Predictions shape: (1905, 10)
Sample prediction (sums to ~100): 100.00


In [14]:
# Use the gold distributions we already built
gold_dist = Y_val_dist
gold_pct = Y_val_dist * 100

# Compute per-label and per-sample MAE for the evaluation cell
per_label_mae = np.abs(gold_pct - pred_pct).mean(axis=0)
per_sample_mae = np.abs(gold_pct - pred_pct).mean(axis=1)

print(f"pred_pct shape: {pred_pct.shape}")
print(f"gold_pct shape: {gold_pct.shape}")
print(f"Ready for evaluation!")

pred_pct shape: (1905, 10)
gold_pct shape: (1905, 10)
Ready for evaluation!


In [15]:
#Evaluate
UO_NAMES = {
    2434: "HU",  # Humaniora
    2436: "JU",  # Juridik
    2438: "LU",  # Lärarutbildning  
    2439: "ME",  # Medicin
    2441: "NA",  # Naturvetenskap
    2442: "SA",  # Samhällsvetenskap
    2444: "TE",  # Teknik
    2445: "VÅ",  # Vård
    2447: "ÖV",  # Övrigt
    2451: "VU"   # Verksamhetsförlagd utbildning
}

print("\nPer-Label MAE (percentage points):")
label_mae_df = pd.DataFrame({
    "uo_code": label_list,
    "uo_name": [UO_NAMES.get(c, f"UO_{c}") for c in label_list],
    "mae_pct": per_label_mae
}).sort_values("mae_pct", ascending=False)
print(label_mae_df.to_string(index=False))

# Per-sample MAE for error analysis
per_sample_mae = np.abs(gold_pct - pred_pct).mean(axis=1)

# Worst predictions
print("\n" + "="*60)
print("WORST PREDICTIONS")
print("="*60)
worst_idx = np.argsort(per_sample_mae)[-10:][::-1]

for idx in worst_idx:
    row = val_ml_export.iloc[idx]
    print(f"\nCourse ID: {row['id']} | MAE: {per_sample_mae[idx]:.2f}")
    print(f"  Gold: {dict(zip(row['labels_uo'], row['labels_pct']))}")
    print(f"  Gold dist: {gold_pct[idx].round(1)}")
    print(f"  Pred dist: {pred_pct[idx].round(1)}")




Per-Label MAE (percentage points):
 uo_code uo_name  mae_pct
    2442      SA 4.222667
    2434      HU 2.198112
    2441      NA 2.152678
    2438      LU 1.809316
    2447      ÖV 1.460524
    2439      ME 0.742094
    2445      VÅ 0.668917
    2444      TE 0.606430
    2436      JU 0.328126
    2451      VU 0.178244

WORST PREDICTIONS

Course ID: 31197 | MAE: 20.00
  Gold: {2447: 100.0}
  Gold dist: [  0.   0.   0.   0.   0.   0.   0.   0. 100.   0.]
  Pred dist: [100.   0.   0.   0.   0.   0.   0.   0.   0.   0.]

Course ID: 28172 | MAE: 20.00
  Gold: {2436: 100.0}
  Gold dist: [  0. 100.   0.   0.   0.   0.   0.   0.   0.   0.]
  Pred dist: [ 0.   0.  99.9  0.   0.   0.   0.   0.   0.   0. ]

Course ID: 26238 | MAE: 20.00
  Gold: {2441: 100.0}
  Gold dist: [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
  Pred dist: [99.   0.   0.   0.   0.   0.   0.   0.   0.9  0. ]

Course ID: 26238 | MAE: 20.00
  Gold: {2441: 100.0}
  Gold dist: [  0.   0.   0.   0. 100.   0.   0.   0.   0

In [16]:
# Save predictions for later comparison with other methods
results_df = val_ml_export[["id", "text", "labels_uo", "labels_pct"]].copy()
results_df["mae"] = per_sample_mae

# Add predicted distribution columns
for i, code in enumerate(label_list):
    results_df[f"pred_{code}"] = pred_pct[:, i]
    results_df[f"gold_{code}"] = gold_pct[:, i]

results_df["gold_primary"] = [label_list[i] for i in np.argmax(Y_val_dist, axis=1)]
results_df["pred_primary"] = [label_list[i] for i in np.argmax(pred_dist, axis=1)]
results_df["primary_match"] = results_df["gold_primary"] == results_df["pred_primary"]

# Save
results_df.to_csv("bert_distributional_predictions.csv", index=False)
print(f"\nSaved predictions to bert_distributional_predictions.csv")

# Summary comparison
print("\n" + "="*60)
print("METHOD COMPARISON SUMMARY")
print("="*60)
print("""
Compare these results with:
1. Binary BERT (post-hoc normalized): 
   - MAE: 2.75, Top-1: 85.1%, Cosine: 0.94

2. Direct Distributional BERT (this model):
   - MAE: {:.2f}, Top-1: {:.1f}%, Cosine: {:.4f}

If distributional training improves metrics, it suggests the model
benefits from learning the actual percentage targets rather than
just presence/absence of labels.
""".format(
    final_metrics["eval_mae_pct"],
    final_metrics["eval_top1_accuracy"] * 100,
    final_metrics["eval_mean_cosine_sim"]
))



Saved predictions to bert_distributional_predictions.csv

METHOD COMPARISON SUMMARY

Compare these results with:
1. Binary BERT (post-hoc normalized): 
   - MAE: 2.75, Top-1: 85.1%, Cosine: 0.94

2. Direct Distributional BERT (this model):
   - MAE: 1.44, Top-1: 92.4%, Cosine: 0.9511

If distributional training improves metrics, it suggests the model
benefits from learning the actual percentage targets rather than
just presence/absence of labels.



In [17]:
# Save the trained model
import torch

# Save model state dict
torch.save(model.state_dict(), "/kaggle/working/bert_distributional_model.pt")

# Save full model (alternative)
torch.save(model, "/kaggle/working/bert_distributional_full.pt")

# Save training args for reproducibility
import json
training_config = {
    "model_path": str(model_dir),
    "num_labels": num_labels,
    "learning_rate": 2e-5,
    "num_epochs": 3,
    "batch_size": 8,
    "final_metrics": final_metrics
}
with open("/kaggle/working/training_config.json", "w") as f:
    json.dump(training_config, f, indent=2)

print("✓ Model saved to /kaggle/working/")

✓ Model saved to /kaggle/working/


In [18]:
# Save predictions (already done)
results_df.to_csv("/kaggle/working/bert_distributional_predictions.csv", index=False)

# Save evaluation metrics
metrics_df = pd.DataFrame([final_metrics])
metrics_df.to_csv("/kaggle/working/bert_distributional_metrics.csv", index=False)

# Save per-label MAE
label_mae_df.to_csv("/kaggle/working/bert_distributional_per_label_mae.csv", index=False)

print("✓ All results saved")

✓ All results saved


To reuse this model, uncomment the code below and run in a new notebook

In [19]:
## In a new notebook
# import torch
# from dist_model import BertForDistributionPrediction

# # Load model
# model = BertForDistributionPrediction(model_path, num_labels=10)
# model.load_state_dict(torch.load("/kaggle/input/your-dataset/bert_distributional_model.pt"))
# model.eval()